# Testy półsyntetyczne: sztuczny odsyp na prawdziwych wskazaniach

Jak dokładnie metody różnicy wskazań liczą masę odsypu, kiedy znamy prawidłowy wynik. Zbiór: baza `2026-10-18 obiekt`, sesja 9, wariant `CURRENT_VARIANT`; obliczenia w `dms_analysis.synthetic`.

Kroki:

1. **Masy i czasy trwania** — z prawdziwych odsypów sesji (metoda `SOURCE_METHOD`, tylko z ujemną różnicą), losowane razem jako para, żeby zachować ich związek.
2. **Próby** — `TRIALS` losowych miejsc we fragmentach bez zdarzeń (`dms_analysis.between_events_fragments`), z zapasem `MARGIN_S` przed początkiem i po końcu, ziarno `TRIAL_SEED`. Próby, w których wycinek zawiera zakłócenie, są pomijane.
3. **Sztuczny odsyp** — od prawdziwych wskazań odejmowana rampa: od początku do końca odsypu liniowo od 0 do pełnej masy, potem pełna masa; wynik zaokrąglony do działki `RESOLUTION_KG`. Bez drgań (decyzja użytkownika z 2026-10-26), więc wynik pokazuje tylko wpływ szumu i dryfu.
4. **Metody** `METHODS`, wyliczona masa porównywana z prawdziwą (błąd = wyliczona − prawdziwa):
   - bez korekcji dryfu: wskazanie przed i po (najbliższa próbka początku i końca albo mediana lub średnia z okna przed początkiem i po końcu), masa = przed − po;
   - z korekcją dryfu („prosta + skok”, „wielomian 3 + skok”, okna `DRIFT_WINDOWS_S`): jeden wielomian podstawy (stopień 1 albo 3) dopasowany naraz do okna N s przed początkiem i okna N s od `SETTLING_S` po końcu, plus skok między oknami (zmienna 0 przed, 1 po); masa = wielkość skoku. Wskazania z czasu odsypu pomijane.
   - Zapas `MARGIN_S` mieści najdłuższe okno korekcji.
5. **Pora dnia** wg zegara RPi: noc `NIGHT_HOURS`, dzień `DAY_HOURS`, reszta „pozostałe” (10–13 i 20–22); „wszystkie” = razem.
6. **Dwa sposoby losowania miejsc:** równomiernie po całym czasie fragmentów (głównie noce, a w nocy prawdziwych odsypów nie ma) i z tej samej godziny co prawdziwy odsyp, z którego wzięto masę i czas (warunki jak przy prawdziwej pracy). Wyniki w osobnych sekcjach do porównania.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

from dms_analysis.between_events_fragments import prepare_samples
from dms_analysis.prepared import load_session
from dms_analysis.signals import CURRENT_VARIANT
from dms_analysis.snapshot import ANALYSIS_SESSION, ANALYSIS_SOURCE
from dms_analysis.synthetic import (
    draw_trials,
    error_figure,
    error_summary,
    measure,
    real_dispenses,
    trial_figure,
    worst_trials,
)

DB_PATH = Path("../data/prepared/obiekt-sesja-9.db")
FIGURE_WIDTH = 1200

In [ ]:
session = load_session(DB_PATH, ANALYSIS_SOURCE, ANALYSIS_SESSION, CURRENT_VARIANT)
samples = prepare_samples(session)

dispenses = real_dispenses(session)
trials = draw_trials(samples, dispenses)
results = measure(samples, trials)

print(f"odsypów źródłowych: {len(dispenses)}, prób: {len(trials)}, policzonych (bez zakłóceń): {results['trial'].nunique()}")
display(trials["part_of_day"].value_counts())

## Błąd metod

Błąd średni pokazuje, czy metoda systematycznie zawyża (> 0) albo zaniża masę; odchylenie standardowe i percentyle |błędu| — rozrzut.

In [ ]:
display(error_summary(results).round(1))

error_figure(results, FIGURE_WIDTH).show()

## Błąd metod: miejsca z godzin prawdziwych odsypów

Te same metody, ale każda próba w losowym miejscu fragmentu bez zdarzeń z tej samej godziny (zegar RPi) co prawdziwy odsyp, z którego wzięto masę i czas trwania. Rozkład pory dnia odpowiada prawdziwym odsypom, więc kolumna „wszystkie” to błąd w warunkach pracy węzła.

In [ ]:
matched_trials = draw_trials(samples, dispenses, same_hour=True)
matched_results = measure(samples, matched_trials)

print(f"prób: {len(matched_trials)}, policzonych (bez zakłóceń): {matched_results['trial'].nunique()}")
display(matched_trials["part_of_day"].value_counts())
display(error_summary(matched_results).round(1))

error_figure(matched_results, FIGURE_WIDTH).show()

## Przegląd prób

Od próby z największym błędem: prawdziwe wskazania i te same ze sztucznym odsypem (zielone pasmo = czas odsypu), w podtytule błąd każdej metody.

`TRIAL_SET` wybiera zestaw: `(trials, results)` (równomiernie) albo `(matched_trials, matched_results)` (godziny prawdziwych odsypów).

In [ ]:
TRIAL_SET = (matched_trials, matched_results)

shown_trials, shown_results = TRIAL_SET
ORDER = worst_trials(shown_results)

widgets.interact(
    lambda position: trial_figure(samples, shown_trials, shown_results, ORDER[position], FIGURE_WIDTH).show(),
    position=widgets.BoundedIntText(value=0, min=0, max=len(ORDER) - 1, description="nr"),
);